# Module 10: Deploy and measure

**Project:** take a trained model out of the notebook and turn it into a **service**: an **API** that other programs can call, and a **web page** that anyone can use. Then **measure** how fast and how expensive it is, and write the one-page idea for your capstone.

**Done when:**
- your model runs behind a **FastAPI** endpoint that you tested with real requests (including bad ones),
- it is online as a **public web page** on Hugging Face Spaces that someone else can open,
- you **documented the latency** (average and 95th percentile over 50 requests) and the **cost per request**,
- you wrote your **capstone idea** (problem, user, why deep learning, data, success metric).

**How to use this notebook**
1. Save your own copy (*File > Save a copy in Drive*). A free CPU runtime is enough.
2. Run cells from top to bottom. Fill in every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
3. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page.

**Which model?** This notebook serves the ticket-urgency classifier from Module 9. You can serve **any model you built** in this course (your Module 1 clothing classifier, your Module 6 search, your Module 9 model): replace the `predict` function in Step 1 and everything else stays the same.

## Step 0: Setup (given)

In [ ]:
import json, time, hashlib, random, threading, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

try:
    import fastapi, uvicorn
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"], check=True)
    import fastapi, uvicorn
import requests, joblib, sklearn
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

random.seed(0); np.random.seed(0)
print("fastapi", fastapi.__version__, "| ready.")

## Step 1: The model to serve (given)

We train the urgency classifier from Module 9 and **save it to a file**, `model.joblib`. A deployed service loads a saved model: it never trains while serving. The function `predict(text)` returns the label and a **score**: the model's probability for the label `urgent`.

*To serve your own model, change `predict` so that it takes text and returns `(label, score)`.*

In [ ]:
def make_ticket_dataset(n=500, seed=0):
    """Synthetic support tickets. label: 'urgent' or 'normal' (about 12% urgent). Two labelers (label, label_b) and a channel (email or chat)."""
    rng = random.Random(seed)
    urgent = ["our whole site is down and customers cannot check out", "we cannot log in and our deadline is in one hour",
              "payment processing is failing for every order", "we lost all our data after the update, please help now",
              "I think someone hacked my account and changed my password", "the app crashes on startup for all our staff",
              "customers are being charged twice, this is critical", "server error 500 on every page since this morning"]
    normal = ["how do I change my billing address", "thanks for the quick reply yesterday", "can you add a dark mode in a future update",
              "what payment methods do you accept", "I would like to know how to export my report", "great product, just a small suggestion about the menu",
              "where can I find my invoice from last month", "do you offer a discount for students", "please update my email address on file",
              "how long does shipping usually take", "I forgot how to reset my password, no rush", "is there a mobile version of the dashboard",
              "login page looks different today, just curious why", "a payment reminder arrived, I wanted to confirm it is correct",
              "the site is a bit slow today but it works"]
    tricky_normal = ["this is not urgent but could you explain the invoice", "URGENT!!! just kidding, I only wanted to say thank you",
                     "no hurry at all, a question about the payment page"]
    borderline = ["login is slow for some of our team", "one customer saw a payment error once", "the export feature fails sometimes",
                  "the page crashed for me twice today"]
    openers = ["", "hi, ", "hello team, ", "good morning, ", "hey, "]
    closers = ["", " thanks", " please advise", " thank you", " regards"]
    out, seen = [], set()
    while len(out) < n:
        r = rng.random()
        if r < 0.10:
            text, lab = rng.choice(urgent), "urgent"; lab_b = lab
        elif r < 0.14:
            text = rng.choice(borderline); lab = "urgent" if rng.random() < 0.5 else "normal"; lab_b = "urgent" if rng.random() < 0.5 else "normal"
        elif r < 0.20:
            text, lab = rng.choice(tricky_normal), "normal"; lab_b = lab
        else:
            text, lab = rng.choice(normal), "normal"; lab_b = lab
        channel = "chat" if rng.random() < 0.4 else "email"
        text = rng.choice(openers) + text + rng.choice(closers)
        if channel == "email":
            text = text[0].upper() + text[1:] + "."
        else:
            text = text.lower().replace(",", "")
            words = text.split()
            if len(words) > 6 and rng.random() < 0.6: text = " ".join(words[:rng.randint(4, 6)])
        if text in seen: 
            text += " " + str(rng.randint(2, 99))
        seen.add(text)
        out.append({"text": text, "label": lab, "label_b": lab_b, "channel": channel})
    return out


data = make_ticket_dataset(1200, seed=0)
texts, labels = [e["text"] for e in data], [e["label"] for e in data]
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(texts, labels)
joblib.dump(model, "model.joblib")

POSITIVE = "urgent"
def predict(text):
    """Return (label, score): the predicted label and the model's probability that the ticket is urgent."""
    probs = model.predict_proba([text])[0]
    score = float(probs[list(model.classes_).index(POSITIVE)])
    return (POSITIVE if score >= 0.5 else "normal"), score

for t in ["our whole site is down and customers cannot check out", "thanks for the quick reply", "login is slow for some of our team"]:
    print(f"{predict(t)[0]:7s} score {predict(t)[1]:.2f} | {t}")
print("Saved model.joblib:", Path("model.joblib").stat().st_size // 1024, "KB")

## Step 2: Log without leaking

A live service should **log what it does**, so that you can see how it behaves, find problems and notice when quality drops. But the text that users send is often **private** (names, passwords, medical details). The rule: **log what you need, not the content**.

Write `make_log_record(text, label, score, latency_ms)`. It returns a dictionary with exactly these keys:
- `ts`: the current UTC time as an ISO string (`datetime.now(timezone.utc).isoformat(timespec="seconds")`),
- `input_chars`: the number of characters in the text,
- `input_sha256`: the **first 12 characters** of the SHA-256 hash of the text (this lets you spot repeated inputs without storing them),
- `label`, `score` (rounded to 4 decimals), `latency_ms` (rounded to 2 decimals).

**The raw text must not appear anywhere in the record.**

*Hint:* `hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]`.

In [ ]:
def make_log_record(text, label, score, latency_ms):
    """A privacy-safe log record: facts about the request, never its content."""
    # YOUR CODE HERE
    return {}

LOG_PATH = Path("requests.jsonl")
def log_request(record):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

### Self-check: the log record

In [ ]:
_text = "My password is hunter2 and my name is Alice"
_rec = make_log_record(_text, "normal", 0.123456, 12.3456)
for _k in ("ts", "input_chars", "input_sha256", "label", "score", "latency_ms"):
    assert _k in _rec, f"The record is missing the key {_k!r}."
_blob = json.dumps(_rec).lower()
assert "hunter2" not in _blob and "alice" not in _blob and "password" not in _blob, "The raw text leaked into the log record. Log facts about the request, not its content."
assert _rec["input_chars"] == len(_text), "input_chars should be the number of characters in the text."
assert _rec["input_sha256"] == hashlib.sha256(_text.encode("utf-8")).hexdigest()[:12], "input_sha256 should be the first 12 hex characters of the SHA-256 of the text."
assert _rec["score"] == 0.1235 and _rec["latency_ms"] == 12.35, f"Round score to 4 decimals and latency to 2 (got {_rec['score']} and {_rec['latency_ms']})."
assert make_log_record(_text, "normal", 0.5, 1)["input_sha256"] == _rec["input_sha256"], "The same text must give the same hash, so repeated inputs can be spotted."
datetime.fromisoformat(_rec["ts"])      # raises an error if ts is not an ISO time
print("make_log_record() check passed: the log shows behaviour, not private content.")

## Step 3: The API

An **API** (application programming interface) lets *other programs* use your model over the network. The web page, the mobile app, and other services all call the same endpoint. **FastAPI** is a popular Python library for building one.

Write `create_app(predict_fn, log_fn=None)`. It builds and returns a FastAPI app with two endpoints:

- **`GET /health`** returns `{"status": "ok"}`. Hosting systems call it to check that your service is alive.
- **`POST /predict`** takes JSON `{"text": "..."}` and returns `{"label": ..., "score": ..., "latency_ms": ...}`.
  - If the text is **empty or only spaces**, or **longer than 2000 characters**, reject the request with `HTTPException(status_code=422, detail=...)` and **do not call the model**.
  - Otherwise time the call to `predict_fn(text)` (`time.perf_counter()`), build the response, and **if `log_fn` is given**, call `log_fn(make_log_record(text, label, score, latency_ms))`.

The request body class `PredictRequest` is given.

In [ ]:
class PredictRequest(BaseModel):
    text: str

def create_app(predict_fn, log_fn=None):
    """Build the FastAPI app with GET /health and POST /predict."""
    app = FastAPI(title="Ticket urgency API")
    # YOUR CODE HERE: the two endpoints
    return app

### Self-check: the API (tested without starting a server)

In [ ]:
_calls, _logs = [], []
def _counting_predict(text):
    _calls.append(text)
    return predict(text)
_client = TestClient(create_app(_counting_predict, log_fn=_logs.append))

_h = _client.get("/health")
assert _h.status_code == 200 and _h.json() == {"status": "ok"}, f"GET /health should return 200 and {{'status': 'ok'}}, but gave {_h.status_code} {_h.text}."

_ok = _client.post("/predict", json={"text": "our whole site is down and customers cannot check out"})
assert _ok.status_code == 200, f"A valid request should return 200 but returned {_ok.status_code}: {_ok.text}"
_body = _ok.json()
assert set(_body) >= {"label", "score", "latency_ms"}, f"The response needs label, score and latency_ms but has {list(_body)}."
assert 0.0 <= _body["score"] <= 1.0 and isinstance(_body["latency_ms"], (int, float)) and _body["latency_ms"] >= 0, "score must be between 0 and 1 and latency_ms a non-negative number."
assert len(_calls) == 1 and len(_logs) == 1, "A valid request must call the model exactly once and write exactly one log record."
assert "whole site" not in json.dumps(_logs[0]), "The log record must not contain the raw text."

for _bad, _why in (({"text": ""}, "an empty text"), ({"text": "    "}, "a text of only spaces"), ({"text": "x" * 2001}, "a text over 2000 characters"), ({}, "a missing text field")):
    _r = _client.post("/predict", json=_bad)
    assert _r.status_code in (400, 422), f"{_why.capitalize()} should be rejected with 422, but the status was {_r.status_code}."
assert len(_calls) == 1 and len(_logs) == 1, "Rejected requests must not reach the model or the log."
def _slow_predict(text):
    time.sleep(0.05)                                   # exactly 50 milliseconds
    return "normal", 0.1
_slow = TestClient(create_app(_slow_predict)).post("/predict", json={"text": "hello there"}).json()
assert 45 <= _slow["latency_ms"] < 1000, f"The model took 50 milliseconds, but latency_ms is {_slow['latency_ms']}. Report milliseconds (seconds x 1000), and time only the model call."
print("create_app() check passed: valid requests work, and bad requests are rejected before they reach the model.")

## Step 4: Run the server and call it like a client (given)

Now start the app as a real server (on this machine, port 8000) and talk to it over the network, the way any other program would. The server runs in the background while you continue.

In [ ]:
app = create_app(predict, log_fn=log_request)

def start_server(app, port=8000):
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(100):
        try:
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=1).status_code == 200:
                return server
        except Exception:
            time.sleep(0.1)
    raise RuntimeError("The server did not start.")

server = start_server(app)
BASE_URL = "http://127.0.0.1:8000"
print("health:", requests.get(f"{BASE_URL}/health").json())
r = requests.post(f"{BASE_URL}/predict", json={"text": "payment processing is failing for every order"})
print("predict:", r.status_code, r.json())
print("bad request:", requests.post(f"{BASE_URL}/predict", json={"text": ""}).status_code)
print("The same call from a terminal would be:")
print("curl -X POST " + BASE_URL + "/predict -H 'Content-Type: application/json' -d '{\"text\": \"our site is down\"}'")

**Look at the API documentation that FastAPI wrote for you.** In a normal browser it lives at `/docs`. Notice that you never wrote it: it comes from your code and the `PredictRequest` class. (In Colab you cannot open `localhost` in a browser tab, so just know that it exists. It will be there when you deploy.)

## Step 5: How fast is it?

Users feel **latency**: the time one request takes. An **average** hides the slow requests, and slow requests are the ones people remember. So we report **percentiles**:

- the **median (p50)**: half the requests are faster than this,
- the **95th percentile (p95)**: 95 of every 100 requests finish within this time. One in twenty is slower.
- the **99th percentile (p99)** and the **maximum**.

Write `summarize_latencies(latencies_ms)`. It returns a dictionary with `n`, `avg`, `p50`, `p95`, `p99` and `max`. Use `np.percentile` for the percentiles.

In [ ]:
def summarize_latencies(latencies_ms):
    """Summarize a list of latencies (milliseconds): n, avg, p50, p95, p99, max."""
    # YOUR CODE HERE
    return {}

### Self-check: latency summary

In [ ]:
_s = summarize_latencies(list(range(1, 101)))        # 1, 2, ..., 100 milliseconds
assert isinstance(_s, dict) and _s.get("n") == 100, "Return a dictionary whose key n is the number of measurements (100 here)."
assert abs(_s["avg"] - 50.5) < 1e-9, f"The average of 1..100 is 50.5 but you got {_s['avg']}."
assert abs(_s["p50"] - 50.5) < 1e-6, f"The median of 1..100 is 50.5 but you got {_s['p50']}."
assert abs(_s["p95"] - 95.05) < 1e-6, f"The 95th percentile of 1..100 is 95.05 (use np.percentile) but you got {_s['p95']}."
assert abs(_s["p99"] - 99.01) < 1e-6 and _s["max"] == 100, "p99 should be 99.01 and max 100."
_skew = summarize_latencies([10] * 95 + [500] * 5)
assert _skew["avg"] < 60 and _skew["p95"] <= 500 and _skew["max"] == 500, "Check your percentiles on a skewed list."
print(f"summarize_latencies() check passed. Note: with 95 fast and 5 slow requests the average is only {_skew['avg']:.0f} ms while the slowest takes {_skew['max']:.0f} ms.")

Now measure the real server with **50 requests**. This cell is given. It times each call from the client side (the whole round trip) and also records the server's own timing.

In [ ]:
def measure_endpoint(base_url, texts, n=50):
    client_ms, server_ms = [], []
    for i in range(n):
        t = texts[i % len(texts)]
        t0 = time.perf_counter()
        r = requests.post(f"{base_url}/predict", json={"text": t})
        client_ms.append((time.perf_counter() - t0) * 1000)
        server_ms.append(r.json()["latency_ms"])
    return client_ms, server_ms

sample_texts = [e["text"] for e in make_ticket_dataset(60, seed=5)]
client_ms, server_ms = measure_endpoint(BASE_URL, sample_texts, n=50)
client_summary, server_summary = summarize_latencies(client_ms), summarize_latencies(server_ms)
for name, s in (("round trip (client)", client_summary), ("model only (server)", server_summary)):
    print(f"{name:20s} avg {s['avg']:6.2f} ms   p50 {s['p50']:6.2f}   p95 {s['p95']:6.2f}   p99 {s['p99']:6.2f}   max {s['max']:6.2f}")

**Question:** how much of the round trip is the model, and how much is everything else (network, JSON, the web server)? Which would you work on first if you needed it faster, and which would you leave alone? Write your answer:

*Your answer:*

## Step 6: What does it cost?

Serving costs money whether or not anyone is calling. You rent a machine **by the hour**, and it only earns its keep while it is **busy**. Three small formulas turn your measurements into a business number. Write them:

**`requests_per_second_capacity(avg_latency_s, workers=1)`**: how many requests per second the service can handle, if each worker handles one request at a time. That is `workers / avg_latency_s`.

**`cost_per_request(avg_latency_s, instance_dollars_per_hour, utilization)`**: the compute cost of one request. The time one request needs is `avg_latency_s`, an hour has 3,600 seconds, so the busy-time cost is `avg_latency_s / 3600 * instance_dollars_per_hour`. But the machine is only busy a fraction `utilization` of the time (between 0 and 1) and you pay for all of it, so divide by `utilization`.

**`monthly_cost(requests_per_day, cost_per_request, fixed_monthly=0.0)`**: 30 days of requests times the cost per request, plus any fixed monthly cost (for example a domain or monitoring).

In [ ]:
def requests_per_second_capacity(avg_latency_s, workers=1):
    # YOUR CODE HERE
    return 0.0

def cost_per_request(avg_latency_s, instance_dollars_per_hour, utilization):
    # YOUR CODE HERE
    return 0.0

def monthly_cost(requests_per_day, cost_per_request_dollars, fixed_monthly=0.0):
    # YOUR CODE HERE
    return 0.0

### Self-check: cost formulas

In [ ]:
assert abs(requests_per_second_capacity(0.05, 2) - 40.0) < 1e-9, f"2 workers at 0.05 s per request handle 40 requests per second, you got {requests_per_second_capacity(0.05, 2)}."
assert abs(requests_per_second_capacity(0.25) - 4.0) < 1e-9, "One worker at 0.25 s per request handles 4 requests per second."
assert abs(cost_per_request(0.05, 0.05, 0.5) - 1.3889e-6) < 1e-9, f"0.05 s at 0.05 dollars per hour with the machine busy half the time costs about 0.0000013889 per request, you got {cost_per_request(0.05, 0.05, 0.5)}."
assert cost_per_request(0.05, 0.05, 0.25) > cost_per_request(0.05, 0.05, 0.5) * 1.99, "A machine that is busy less often costs more per request: divide by utilization."
assert abs(monthly_cost(10_000, 0.00002, 20.0) - 26.0) < 1e-9, f"10,000 requests a day for 30 days at 0.00002 each is 6.0, plus 20.0 fixed = 26.0. You got {monthly_cost(10_000, 0.00002, 20.0)}."
assert monthly_cost(0, 0.01) == 0.0, "No requests and no fixed cost means no cost."
print("Cost formulas check passed.")

Now plug in your **measured** latency and a realistic **machine price**. The price below is a placeholder: look up the real hourly price of the machine you would rent (a small cloud instance costs a few cents to a few dollars per hour). This cell is given.

In [ ]:
INSTANCE_DOLLARS_PER_HOUR = 0.05     # PLACEHOLDER: look up the real price of your machine
WORKERS = 1
FIXED_MONTHLY = 0.0                  # for example a domain name or monitoring

avg_latency_s = server_summary["avg"] / 1000
capacity = requests_per_second_capacity(avg_latency_s, WORKERS)
print(f"Measured model latency {server_summary['avg']:.2f} ms  ->  one worker can handle about {capacity:,.0f} requests per second")
print("\nCost per request at different utilization (how busy the machine is):")
for u in (0.05, 0.25, 0.5, 0.9):
    c = cost_per_request(avg_latency_s, INSTANCE_DOLLARS_PER_HOUR, u)
    print(f"  utilization {u:4.0%}: ${c:.8f} per request   ${1000 * c:.5f} per 1,000   ${monthly_cost(100_000, c, FIXED_MONTHLY):,.2f} per month at 100,000 requests a day")

**Question:** at low traffic, what is your real monthly cost, and what does it depend on: your model, or the fact that the machine is on? When would you scale to a second machine? What one number from the table above matters most to a founder?

*Your answer:*

## Step 7: Put it on the web (Hugging Face Spaces)

An API is for programs. People need a **web page**. **Hugging Face Spaces** hosts a small app for free and gives it a public address. The app uses **Gradio**, which turns a Python function into a web page in a few lines.

This cell is given. It writes the three files a Space needs into a folder `space/`: `app.py` (the web page), `requirements.txt` (the libraries), and your `model.joblib`. It also zips them so that you can download them.

In [ ]:
Path("space").mkdir(exist_ok=True)
import shutil; shutil.copy("model.joblib", "space/model.joblib")

Path("space/app.py").write_text('''import gradio as gr
import joblib

model = joblib.load("model.joblib")
CLASSES = list(model.classes_)

def classify(text):
    """Return the probability of each class for the given text."""
    if not text or not text.strip():
        return {c: 0.0 for c in CLASSES}
    probs = model.predict_proba([text.strip()[:2000]])[0]
    return {c: float(p) for c, p in zip(CLASSES, probs)}

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=3, label="Support ticket"),
    outputs=gr.Label(num_top_classes=2, label="Urgency"),
    title="Ticket urgency classifier",
    description="Paste a support ticket and see how urgent the model thinks it is.",
    examples=["our whole site is down and customers cannot check out", "thanks for the quick reply", "how do I change my billing address"],
)

if __name__ == "__main__":
    demo.launch()
''')
Path("space/requirements.txt").write_text(f"gradio\nscikit-learn=={sklearn.__version__}\njoblib\n")   # pinned: the model must be loaded by the same scikit-learn version that saved it

shutil.make_archive("space", "zip", "space")
print("Files for your Space:", sorted(p.name for p in Path("space").iterdir()))
try:
    from google.colab import files
    files.download("space.zip")
except ImportError:
    print("Download space.zip from the file panel.")

### Self-check: the Space files

In [ ]:
import ast
_src = Path("space/app.py").read_text()
ast.parse(_src)                                   # raises an error if app.py has a syntax error
assert "gradio" in _src and "gr.Interface" in _src and "joblib.load" in _src, "app.py should build a gr.Interface around the loaded model."
_req = Path("space/requirements.txt").read_text().lower()
assert all(lib in _req for lib in ("gradio", "scikit-learn", "joblib")), "requirements.txt must list gradio, scikit-learn and joblib."
assert Path("space/model.joblib").exists() and Path("space/model.joblib").stat().st_size > 1000, "model.joblib is missing from the space folder."
_loaded = joblib.load("space/model.joblib")
assert list(_loaded.predict(["our whole site is down and customers cannot check out"])) == ["urgent"], "The saved model does not behave as expected."
print("Space files check passed.")

**Now deploy it** (about 10 minutes):
1. Create a free account at huggingface.co, then **New Space**. Choose the **Gradio** SDK, free CPU hardware, and public visibility.
2. In the **Files** tab choose *Add file, Upload files*, and upload `app.py`, `requirements.txt` and `model.joblib` (unzip `space.zip` on your computer first).
3. Wait a few minutes while it builds. Open the address (it looks like `https://your-name-your-space.hf.space`) in a private window and try it.
4. Ask a friend to open it. **It must work for someone else.**

Then paste the address into `PUBLIC_URL`.

In [ ]:
PUBLIC_URL = ""      # YOUR CODE HERE: the https:// address of your Space, for example "https://your-name-ticket-urgency.hf.space"

### Self-check: the public URL

In [ ]:
assert PUBLIC_URL.startswith("https://"), "PUBLIC_URL must be the full https:// address of your Space."
assert "hf.space" in PUBLIC_URL or "huggingface.co/spaces" in PUBLIC_URL, "A Hugging Face Space address contains hf.space (or huggingface.co/spaces/...)."
try:
    _resp = requests.get(PUBLIC_URL, timeout=30)
    assert _resp.status_code == 200, f"Your Space answered with status {_resp.status_code}. Is it running (not sleeping, not on the build log)? Is it set to public?"
except requests.RequestException as e:
    raise AssertionError(f"Could not reach {PUBLIC_URL}: {e}. Open it in a browser and check that the Space is public and finished building.")
print("Public URL check passed: your Space is online and answers requests.")

## Step 8: Your capstone idea

In the last two modules you will build **your own AI startup prototype**. Write the one-page idea now, while this module's lessons are fresh. Fill in each field with **at least two sentences** (40 characters or more):

- `problem`: whose problem, and how painful is it today?
- `user`: who exactly will use it, and how (an app, an API, an email)?
- `why_deep_learning`: why is a learned model the right tool, and **what simpler alternative** (rules, search, a spreadsheet, a non-learned baseline) would you try first, and why might it fall short?
- `data_source`: where will your data come from, and how will you label it and keep a fixed test set (Module 9)?
- `success_metric`: the number you will measure on your test set, and the value that means "good enough". Include cost or latency if they matter.
- `risks`: what can go wrong (wrong answers, bias, privacy, cost)?

In [ ]:
CAPSTONE_IDEA = {
    "problem": "",            # YOUR CODE HERE
    "user": "",
    "why_deep_learning": "",
    "data_source": "",
    "success_metric": "",
    "risks": "",
}

### Self-check: the capstone idea

In [ ]:
_keys = ["problem", "user", "why_deep_learning", "data_source", "success_metric", "risks"]
assert set(CAPSTONE_IDEA) >= set(_keys), f"CAPSTONE_IDEA needs the keys {_keys}."
for _k in _keys:
    assert isinstance(CAPSTONE_IDEA[_k], str) and len(CAPSTONE_IDEA[_k].strip()) >= 40, f"Write at least two sentences (40 characters or more) for {_k!r}."
_why = CAPSTONE_IDEA["why_deep_learning"].lower()
assert any(w in _why for w in ("baseline", "rule", "simpler", "simple", "without", "spreadsheet", "keyword")), "In why_deep_learning, say which simpler alternative you would try first."
assert any(ch.isdigit() for ch in CAPSTONE_IDEA["success_metric"]), "Your success metric must contain a number."
print("Capstone idea check passed.")

## Step 9: The deployment record (given)

This cell collects everything into one file, `deployment.json`: your public URL, your latency, your cost, and your capstone idea. Submit it with your notebook.

In [ ]:
c_req = cost_per_request(avg_latency_s, INSTANCE_DOLLARS_PER_HOUR, 0.5)
record = {"public_url": PUBLIC_URL,
          "latency_ms_round_trip": {k: round(v, 2) for k, v in client_summary.items()},
          "latency_ms_model_only": {k: round(v, 2) for k, v in server_summary.items()},
          "capacity_requests_per_second": round(capacity, 1),
          "assumptions": {"instance_dollars_per_hour": INSTANCE_DOLLARS_PER_HOUR, "workers": WORKERS, "utilization": 0.5},
          "cost_per_1000_requests_dollars": round(1000 * c_req, 6),
          "monthly_cost_at_100k_requests_per_day_dollars": round(monthly_cost(100_000, c_req, FIXED_MONTHLY), 2),
          "capstone_idea": CAPSTONE_IDEA}
Path("deployment.json").write_text(json.dumps(record, indent=2))
print(json.dumps({k: record[k] for k in ("public_url", "capacity_requests_per_second", "cost_per_1000_requests_dollars")}, indent=2))
print("Saved deployment.json")

## Stretch challenge (optional): make it faster

**Caching.** Real users often send the same input twice. Wrap `predict` with a cache so that a repeated text is answered from memory instead of recomputed: `functools.lru_cache(maxsize=1000)` on a function of the text. Then measure again with `sample_texts` that **repeat** (for example only 10 different texts used for 50 requests) and compare the p50 and p95 before and after. Report the speed-up. What is the risk of caching (a model that you update still answers with old results; private text kept in memory)? How would you limit it?

**Load.** Send requests from 10 threads at once (`concurrent.futures.ThreadPoolExecutor`). Does the p95 change? What does that tell you about the capacity formula?

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-10/README.md`:
1. What you deployed, and your public URL
2. Your p95 latency and your cost per 1,000 requests
3. The one change that would most improve speed or cost, and your capstone idea in one sentence

Then download this notebook (*File > Download > .ipynb*), push it to `module-10` in your portfolio repo together with `deployment.json` and the `space/app.py` file (do **not** push private data or keys), and submit the link in Moodle.